# Source coding

> A payload as bits, and back: integers, codebook indices, quantized real numbers.

In [ ]:
#| default_exp dsp.source_coding

In [ ]:
#| hide
from nbdev.showdoc import *

`comm_core.dsp` holds the signal processing of a link-level transmission, apart from the network's components:

| module | does | |
|---|---|---|
| `source_coding` | payload ↔ bits | `to_bits`, `from_bits`, `Codec`: `IndexCodec`, `UniformQuantizer` |
| `modulation` | bits ↔ symbols | `Modulation`: `BPSK`, `SionnaModulation` |
| `signal` | symbols → what the receiver gets, `y = h √P x + n` | `awgn`, `apply_channel` |

Each piece is a plain function or a small class on numpy arrays or torch tensors (on any device), with any batch
dimensions in front. None of them needs a `Network`, so training code can use them on whole batches of links. The
link-level receiver block, `BitLevelLink` (see `radio.link.channel`), chains them for a network's transmissions.

In [ ]:
#| export
from __future__ import annotations

import math
from abc import ABC, abstractmethod

import numpy as np

from comm_core.block import _is_torch

In [ ]:
#| export
def _arange(n: int, like):
    "0, ..., n-1: a torch tensor on `like`'s device if `like` is one, else a numpy array."
    if _is_torch(like):
        import torch
        return torch.arange(n, device=like.device)
    return np.arange(n)


def _as_array(x, dtype=None):
    "Tensors stay tensors; anything else becomes a numpy array (at least 1-d)."
    return x if _is_torch(x) else np.atleast_1d(np.asarray(x, dtype=dtype))

## Bits

Integers and their bits, least significant bit first. Both functions work on the last dimension.

In [ ]:
#| export
def to_bits(
    indices, # Integers, (..., M)
    bits_per_index: int, # Bits per integer
):
    "The bits of `indices`: (..., M * bits_per_index), least significant bit first."
    shifts = _arange(bits_per_index, indices)
    bits = (indices[..., None] >> shifts) & 1
    return bits.reshape(*indices.shape[:-1], -1)


def from_bits(
    bits, # 0/1 or booleans, (..., M * bits_per_index)
    bits_per_index: int, # Bits per integer
):
    "The integers of `bits` (the inverse of `to_bits`): (..., M)."
    weights = 2 ** _arange(bits_per_index, bits)
    return (bits.reshape(*bits.shape[:-1], -1, bits_per_index) * weights).sum(-1)

In [ ]:
x = np.array([[0, 1, 5, 255], [128, 7, 2, 3]])
b = to_bits(x, 8)
assert b.shape == (2, 32) and b[0, 8:16].tolist() == [1, 0, 0, 0, 0, 0, 0, 0]       # 1, least significant bit first
assert (from_bits(b, 8) == x).all() and (from_bits(b.astype(bool), 8) == x).all()

## Codecs

A `Codec` turns a payload into bits and back: source coding. `IndexCodec` is for discrete messages, e.g. indices
into a VQ codebook. `UniformQuantizer` is for real numbers, e.g. latents or positions. Protecting the bits against
the channel's errors is channel coding, a different step: the LDPC code of `SionnaLink` (see `radio.link.channel`).

In [ ]:
#| export
class Codec(ABC):
    "Payload <-> bits."

    @abstractmethod
    def encode(self, payload):
        "The payload's bits: (..., num_bits) of 0/1."

    @abstractmethod
    def decode(self, bits):
        "The payload of `bits`, which may have bit errors."


class IndexCodec(Codec):
    "Indices into a codebook (e.g. a VQ-VAE's), with ceil(log2(codebook_size)) bits each."

    def __init__(
        self,
        codebook_size: int = 256, # Number of codewords
    ):
        self.codebook_size = codebook_size
        self.bits_per_index = max(1, math.ceil(math.log2(codebook_size)))

    def encode(self, payload):
        "`payload`: integers in [0, codebook_size), (..., M)."
        x = _as_array(payload)
        if _is_torch(x):
            integer = not (x.is_floating_point() or x.is_complex()) and str(x.dtype) != 'torch.bool'
        else:
            integer = np.issubdtype(x.dtype, np.integer)
        if not integer:
            raise TypeError(f"IndexCodec encodes integers, not {x.dtype}")
        if (x < 0).any() or (x >= self.codebook_size).any():
            raise ValueError(f"indices must be in [0, {self.codebook_size})")
        return to_bits(x, self.bits_per_index)

    def decode(self, bits):
        "Indices (..., M). Bit errors can give indices >= codebook_size if it is not a power of 2."
        return from_bits(bits, self.bits_per_index)


class UniformQuantizer(Codec):
    "Real numbers in [low, high], each rounded to one of 2**bits evenly spaced levels (values outside are clipped)."

    def __init__(
        self,
        bits: int = 8, # Bits per number
        low: float = -1.0, # Smallest value
        high: float = 1.0, # Largest value
    ):
        self.bits, self.low, self.high = bits, low, high
        self.levels = 2 ** bits - 1

    def encode(self, payload):
        x = _as_array(payload, dtype=float)
        q = ((x.clip(self.low, self.high) - self.low) / (self.high - self.low) * self.levels).round()
        return to_bits(q.long() if _is_torch(q) else q.astype(np.int64), self.bits)

    def decode(self, bits):
        return self.low + from_bits(bits, self.bits) * ((self.high - self.low) / self.levels)

In [ ]:
c = IndexCodec(256)
assert c.bits_per_index == 8 and c.encode([3, 200]).shape == (16,) and c.decode(c.encode([3, 200])).tolist() == [3, 200]
assert IndexCodec(200).bits_per_index == 8 and IndexCodec(2).bits_per_index == 1
for bad, err in (([1.5], TypeError), ([True], TypeError), ([256], ValueError), ([-1], ValueError)):
    try:
        c.encode(bad); raise AssertionError(bad)
    except err:
        pass
q = UniformQuantizer(bits=8, low=-1, high=1)
x = np.array([-1.0, -0.3, 0.0, 0.77, 1.0, 3.0])
y = q.decode(q.encode(x))
assert y.shape == x.shape and np.abs(y - x.clip(-1, 1)).max() <= 1 / 255 + 1e-12      # within half a step (2/255 / 2)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()